# FURAX Mapmaking Tutorial

This tutorial covers FURAX's mapmaking module end-to-end.

**Parts:**
- **Part 1** — How to run the mapmaker (data loading, config, results, plotting)
- **Part 2** — How the mapmaker works (acquisition, noise models, PCG solver, gap filling)
- **Part 3** — Mapmaking algorithms (Binned, ML, ATOP, TwoStep, custom)

**Prerequisites:**
```bash
pip install -e '.[mapmaking,so,toast]'
```

The code cells that load real data reference `tests/data/` which ships with the repository.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# furax requires 64-bit floats — enable before any JAX computation
jax.config.update('jax_enable_x64', True)

# Paths to the bundled test data
REPO_ROOT = Path('../../..').resolve()   # adjust if running from elsewhere
TEST_DATA = REPO_ROOT / 'tests' / 'data'
SOTODLIB_DATA = TEST_DATA / 'sotodlib'
TOAST_DATA = TEST_DATA / 'toast'

print('JAX devices:', jax.devices())
print('Test data:', TEST_DATA)

---
# Part 1: Running FURAX Mapmaker

This part walks through the complete user workflow:
1. Load observation data
2. Configure the mapmaker
3. Run mapmaking and save outputs
4. Plot results
5. Multi-observation mapmaking

## 1.1 Loading Observation Data

FURAX wraps external data containers behind a uniform `AbstractGroundObservation` interface.
Every concrete subclass exposes the same set of methods regardless of the backend:

| Method | Returns | Description |
|--------|---------|-------------|
| `get_tods()` | `(ndets, nsamps)` | Raw time-ordered data |
| `get_boresight_quaternions()` | `(nsamps, 4)` | Boresight pointing quaternions |
| `get_detector_quaternions()` | `(ndets, 4)` | Detector offset quaternions |
| `get_hwp_angles()` | `(nsamps,)` | HWP rotation angles (rad) |
| `get_timestamps()` | `(nsamps,)` | Unix timestamps (s) |
| `get_azimuth()` | `(nsamps,)` | Boresight azimuth (rad) |
| `get_elevation()` | `(nsamps,)` | Boresight elevation (rad) |
| `get_noise_model()` | `NoiseModel \| None` | Pre-fitted noise model |
| `get_sample_mask()` | `(ndets, nsamps)` bool | True = valid sample |

Two backends are currently supported: **sotodlib** and **TOAST3**.

### sotodlib

`SOTODLibObservation` wraps a sotodlib `AxisManager`. Three construction paths:

In [ ]:
from furax.interfaces.sotodlib import SOTODLibObservation, LazySOTODLibObservation
from furax.mapmaking.config import SotodlibConfig

# --- Option A: load directly from an HDF5 file ---
sotodlib_obs = SOTODLibObservation.from_file(SOTODLIB_DATA / 'test_obs.h5')

# --- Option B: load with sotodlib-specific settings ---
sotodlib_cfg = SotodlibConfig(
    site='so_sat3',          # observatory site tag
    weather='toco',          # atmospheric model tag
    demodulated=False,       # set True for demodulated (post-HWP) data
    wobble_correction=False, # apply HWP wobble correction
    noise_source='preprocess',  # 'preprocess' or 'mapmaking' noise fits
)
sotodlib_obs = SOTODLibObservation.from_file(
    SOTODLIB_DATA / 'test_obs.h5',
    sotodlib_config=sotodlib_cfg,
)

# --- Option C: from sotodlib preprocessing pipeline ---
# sotodlib_obs = SOTODLibObservation.from_preprocess(
#     preprocess_config='preprocess.yaml',
#     observation_id='obs_1714550584_satp3_1111111',
#     detector_selection={'wafer_slot': 'ws0', 'wafer.bandpass': 'f150'},
# )

print(f'n_detectors : {sotodlib_obs.n_detectors}')
print(f'n_samples   : {sotodlib_obs.n_samples}')
print(f'sample_rate : {sotodlib_obs.sample_rate:.1f} Hz')
print(f'duration    : {sotodlib_obs.n_samples / sotodlib_obs.sample_rate:.1f} s')

### TOAST3

`ToastObservation` wraps a TOAST3 `toast.Data` container.

In [ ]:
import toast
from toast.ops.load_hdf5 import LoadHDF5

from furax.interfaces.toast import ToastObservation, LazyToastObservation

# --- Option A: load from HDF5 file ---
toast_obs = ToastObservation.from_file(TOAST_DATA / 'test_obs.h5')

# --- Option B: wrap an existing toast.Data object ---
# data = toast.Data()
# loader = LoadHDF5(files=[str(TOAST_DATA / 'test_obs.h5')])
# loader.apply(data)
# toast_obs = ToastObservation(data=data, observation_index=0)

print(f'n_detectors : {toast_obs.n_detectors}')
print(f'n_samples   : {toast_obs.n_samples}')
print(f'sample_rate : {toast_obs.sample_rate:.1f} Hz')

### Inspecting the observation

In [ ]:
# Use toast_obs for the rest of Part 1
obs = toast_obs

tod   = obs.get_tods()                 # (ndets, nsamps)
times = obs.get_elapsed_times()        # (nsamps,)  — seconds since start
az    = np.degrees(obs.get_azimuth())  # boresight azimuth in degrees
el    = np.degrees(obs.get_elevation())

fig, axs = plt.subplots(1, 2, figsize=(12, 3))

axs[0].set_title('TOD — first 4 detectors')
for i in range(min(4, obs.n_detectors)):
    axs[0].plot(times[:2000], tod[i, :2000], lw=0.5, alpha=0.8)
axs[0].set_xlabel('Time [s]'); axs[0].set_ylabel('Signal')

axs[1].set_title('Boresight scanning pattern')
axs[1].plot(az, el, ',', ms=0.5, alpha=0.3)
axs[1].set_xlabel('Azimuth [deg]'); axs[1].set_ylabel('Elevation [deg]')

plt.tight_layout()
plt.show()

## 1.2 Creating a Mapmaker from Config

All mapmaking options are collected in `MapMakingConfig`. The key sub-configs are:

| Sub-config | Controls |
|------------|----------|
| `landscape` | Output pixelisation (HEALPix or WCS) and Stokes components |
| `noise` | Noise model type (white / 1/f) and fitting options |
| `pointing` | On-the-fly vs pre-computed, interpolation scheme |
| `solver` | PCG tolerance and iteration budget |
| `gaps` | Gap-filling and nested PCG options |
| `templates` | Optional systematics templates (polynomial, HWP-synchronous, ...) |
| `sotodlib` | sotodlib-specific options (site, demodulation, ...) |

The `method` field selects the mapmaking algorithm:

| `Methods` enum | Algorithm | Noise requirement |
|----------------|-----------|------------------|
| `Methods.BINNED` | Binned  | `noise.white=True` |
| `Methods.MAXL` | Maximum Likelihood (PCG) | `noise.white=False` |
| `Methods.TWOSTEP` | Two-step (templates + binning) | `noise.white=True` |
| `Methods.ATOP` | ATOP (QU only) | `noise.white=True` |

In [ ]:
from furax.mapmaking import MapMakingConfig
from furax.mapmaking.config import (
    GapsConfig,
    HealpixConfig,
    LandscapeConfig,
    Methods,
    NoiseFitConfig,
    NoiseConfig,
    PointingConfig,
    SkyPatch,
    SolverConfig,
    WCSConfig,
)
from furax.obs.landscapes import ProjectionType

# ── HEALPix binned mapmaker ─────────────────────────────────────────────────
binned_config = MapMakingConfig(
    method=Methods.BINNED,

    # Output sky map: IQU on HEALPix nside=64
    landscape=LandscapeConfig(
        stokes='IQU',
        healpix=HealpixConfig(nside=64),
    ),

    # Noise model: white (diagonal) noise, fit from the TOD
    noise=NoiseConfig(
        white=True,          # True → binned; False → ML (Toeplitz)
        fit_from_data=True,  # estimate sigma from Welch PSD
        fitting=NoiseFitConfig(
            nperseg=512,         # Welch FFT window length
            mask_hwp_harmonics=True,   # mask 1f/2f/4f HWP lines
        ),
    ),

    # Pointing: computed on-the-fly (memory-efficient)
    pointing=PointingConfig(
        on_the_fly=True,
        chunk_size=32,              # detectors processed per chunk
        interpolation='nearest',    # or 'bilinear'
    ),

    double_precision=True,  # use float64
    scanning_mask=False,    # discard turnaround samples
    sample_mask=False,      # additionally discard glitch-flagged samples
    hits_cut=1e-2,          # drop pixels with < 1% of peak hits
    cond_cut=1e-2,          # drop poorly conditioned pixels
)

print(binned_config._to_yaml()[:600], '...')

In [ ]:
# ── WCS (CAR) landscape example ─────────────────────────────────────────────
# Useful for small-patch observations.
# Three ways to specify the footprint:
#   1. explicit patch  (shown below)
#   2. auto-scan: omit patch and geometry_file — footprint is computed from data
#   3. geometry_file: read shape + WCS from an existing FITS map

wcs_config = MapMakingConfig(
    method=Methods.BINNED,
    landscape=LandscapeConfig(
        stokes='IQU',
        wcs=WCSConfig(
            projection=ProjectionType.CAR,
            resolution=4.0,  # arcmin/pixel
            patch=SkyPatch(
                center=(30.0, -10.0),  # (RA, Dec) in degrees
                width=20.0,            # degrees
                height=10.0,           # degrees
            ),
        ),
    ),
    noise=NoiseConfig(white=True, fit_from_data=True),
    pointing=PointingConfig(on_the_fly=True),
)
print('WCS config OK')

In [ ]:
# Configs can be saved and loaded as YAML — useful for batch jobs
binned_config.dump_yaml('/tmp/mapmaking_config.yaml')

# Load back
loaded = MapMakingConfig.load_yaml('/tmp/mapmaking_config.yaml')
print('Loaded method:', loaded.method)
print('Loaded nside :', loaded.landscape.healpix.nside)

In [ ]:
from furax.mapmaking.mapmaker import MapMaker

# MapMaker.from_config() dispatches to the right class based on method
mapmaker = MapMaker.from_config(binned_config)
print(type(mapmaker).__name__)  # BinnedMapMaker

# You can also load a mapmaker directly from a YAML file
# mapmaker = MapMaker.from_yaml('/tmp/mapmaking_config.yaml')

## 1.3 Running Mapmaking and Saving Results

The single-observation mapmaker accepts any `AbstractGroundObservation`.

- `mapmaker.make_map(obs)` → runs and returns a `dict`
- `mapmaker.run(obs, out_dir)` → same, but also writes outputs to disk

In [ ]:
# Run the binned mapmaker on the toast observation
results = mapmaker.run(toast_obs, out_dir='/tmp/furax_maps/')

print('Output keys:', list(results.keys()))
print('map shape   :', results['map'].shape)      # (3, n_pixels) — I, Q, U
print('weights shape:', results['weights'].shape) # (n_pixels, 3, 3) — per-pixel icov

In [ ]:
# What each key contains
# 'map'          : np.ndarray (3, n_pixels) — I, Q, U sky maps
# 'weights'      : np.ndarray (n_pixels, 3, 3) — per-pixel inverse noise covariance
# 'wcs'          : astropy.wcs.WCS (only for WCS landscapes)
# 'noise_fit'    : np.ndarray (ndets, n_params) — fitted noise model (if fit_from_data=True)
# 'proj_map'     : projected map TOD (if debug=True)

# For the ML mapmaker:
# 'weights_uncut': weights before pixel selection (all pixels, not just valid ones)
# 'template_*'   : template amplitude arrays (if templates are configured)

print('IQU map RMS:', [float(np.std(results['map'][i][results['weights'][:, i, i] > 0]))
                       for i in range(3)])

## 1.4 Plotting Results

In [ ]:
import healpy as hp

iqu_map = results['map']           # (3, 12 * nside^2)
weights_ii = results['weights'][:, 0, 0]  # I-component weight
nside = binned_config.landscape.healpix.nside

# Mask unobserved pixels
mask = weights_ii > 0
plot_map = iqu_map.copy()
plot_map[:, ~mask] = hp.UNSEEN

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for i, (stoke, ax) in enumerate(zip('IQU', axs)):
    hp.cartview(
        plot_map[i],
        title=stoke,
        hold=True,
        sub=(1, 3, i + 1),
        min=np.nanpercentile(plot_map[i][mask], 1),
        max=np.nanpercentile(plot_map[i][mask], 99),
        cmap='RdBu_r',
        fig=fig.number,
    )
plt.suptitle('Binned mapmaker output (nside=64)', y=1.02)
plt.show()

In [ ]:
# For WCS (pixell) maps, use the pixell_utils helpers
from furax.mapmaking.pixell_utils import ndmap_from_wcs_landscape, plot_ndmap

# wcs_results = MapMaker.from_config(wcs_config).run(obs)
# ndmap = ndmap_from_wcs_landscape(wcs_results['map'], wcs_landscape)
# fig, ax = plot_ndmap(ndmap[0], title='I')

print('(WCS plotting requires running with a WCS landscape — uncomment above)')

## 1.5 Multi-Observation Mapmaking with `MultiObservationMapMaker`

`MultiObservationMapMaker` co-adds any number of observations into a single map.
It uses the same `MapMakingConfig` but works with **lazy** observation wrappers that
load data on demand — keeping memory usage bounded.

Key differences from the single-obs `MapMaker`:

| | `MapMaker` | `MultiObservationMapMaker` |
|--|--|--|
| Input | `AbstractGroundObservation` | `list[AbstractLazyObservation]` |
| Output | `dict` | `MapMakingResults` dataclass |
| Method | `.run(obs, out_dir)` | `.run(out_dir)` |

`MapMakingResults` has structured attributes:
- `.map` — `StokesIQU` (or `StokesQU`, etc.) pytree of sky arrays
- `.icov` — `(n_stokes, n_stokes, *pixels)` inverse noise covariance
- `.hit_map` — `(*pixels,)` hit count per pixel
- `.solver_stats` — dict with `num_steps`, `max_steps`
- `.landscape` — `StokesLandscape` with pixelisation info

In [ ]:
from furax.mapmaking import MultiObservationMapMaker
from furax.interfaces.toast import LazyToastObservation
from furax.interfaces.sotodlib import LazySOTODLibObservation

# Lazy wrappers: files are opened only when data is actually needed
toast_lazy_obs = [
    LazyToastObservation(TOAST_DATA / 'test_obs.h5'),
    LazyToastObservation(TOAST_DATA / 'test_obs.h5'),  # simulating two observations
]

# Same config structure as single-obs
multi_config = MapMakingConfig(
    method=Methods.BINNED,
    landscape=LandscapeConfig(
        stokes='IQU',
        healpix=HealpixConfig(nside=64),
    ),
    noise=NoiseConfig(
        white=True,
        fit_from_data=True,
        fitting=NoiseFitConfig(nperseg=512),
    ),
    pointing=PointingConfig(on_the_fly=True),
)

maker = MultiObservationMapMaker(toast_lazy_obs, config=multi_config)

# run() optionally saves all products to out_dir
results_multi = maker.run(out_dir='/tmp/furax_multi/')

print('Type :', type(results_multi).__name__)
print('Map  :', type(results_multi.map).__name__, '→ .i .q .u arrays')
print('icov :', results_multi.icov.shape)  # (3, 3, n_pixels)
print('hits :', results_multi.hit_map.shape)
print('Steps:', results_multi.solver_stats['num_steps'])

In [ ]:
# sotodlib lazy observations work identically
sotodlib_lazy_obs = [
    LazySOTODLibObservation(SOTODLIB_DATA / 'test_obs.h5'),
    LazySOTODLibObservation(SOTODLIB_DATA / 'test_obs_2.h5'),
]

# For demodulated data, pass SotodlibConfig
# sotodlib_lazy_obs = [
#     LazySOTODLibObservation(path, sotodlib_config=SotodlibConfig(demodulated=True))
#     for path in [file1, file2, ...]
# ]

# WCS footprint can be determined automatically by scanning all observations
auto_wcs_config = MapMakingConfig(
    method=Methods.BINNED,
    landscape=LandscapeConfig(
        stokes='IQU',
        wcs=WCSConfig(
            projection=ProjectionType.CAR,
            resolution=60.0,  # arcmin — coarse for the test data
            # no patch/geometry_file: footprint scanned from data
        ),
    ),
    noise=NoiseConfig(white=True, fit_from_data=True, fitting=NoiseFitConfig(nperseg=512)),
    pointing=PointingConfig(on_the_fly=True),
)

maker_so = MultiObservationMapMaker(sotodlib_lazy_obs, config=auto_wcs_config)
results_so = maker_so.run()
print('SO results map shape:', results_so.map.shape)

---
# Part 2: How the FURAX Mapmaker Works

This part goes under the hood: acquisition operators, noise models, the mapmaking
equation, PCG solver, gap filling, and nested PCG.

## 2.1 The Acquisition Operator

The **acquisition operator** $\mathbf{H}$ maps sky Stokes maps to time-ordered data:

$$d = \mathbf{H} \, s + n$$

For a detector at pointing angle $\psi_t$ and HWP angle $\chi_t$, a single sample reads:

$$d_t = \tfrac{1}{2}\!\left[ I_p + \cos(\phi_t)\,Q_p + \sin(\phi_t)\,U_p \right]$$

where $p$ is the sky pixel and $\phi_t = 2(2\chi_t + \psi_t - 2\gamma)$ with $\gamma$ the
detector orientation angle (without HWP: $\phi_t = 2\psi_t$).

### Operator chain

The full acquisition decomposes as:

```
H = LinearPolarizer @ QURotation(gamma) @ HWP(chi) @ Pointing
```

Each factor is a FURAX `AbstractLinearOperator` supporting `.mv()` (forward) and
`.T` (adjoint).

In [ ]:
from furax.obs.landscapes import HealpixLandscape

# Toy problem: random quaternions, small sky
key = jax.random.PRNGKey(0)
NSIDE, NDET, NSAMP = 4, 3, 100

def rand_unit_quats(key, shape):
    q = jax.random.normal(key, (*shape, 4))
    return q / jnp.linalg.norm(q, axis=-1, keepdims=True)

k1, k2, k3, k4 = jax.random.split(key, 4)
qbore = rand_unit_quats(k1, (NSAMP,))   # (NSAMP, 4)
qdet  = rand_unit_quats(k2, (NDET,))    # (NDET,  4)
hwp_angles = jax.random.uniform(k4, (NSAMP,), maxval=jnp.pi)

landscape = HealpixLandscape(nside=NSIDE, stokes='IQU', dtype=jnp.float64)
sky_true  = landscape.normal(k3)  # StokesIQU with .i .q .u arrays of shape (12*NSIDE^2,)

print('Sky shape:', sky_true.i.shape)
print('Sky type :', type(sky_true).__name__)

### Pointing operator

`PointingOperator` converts boresight + detector quaternions into map look-up indices
and applies the coordinate rotation that takes the sky frame into the telescope frame.

In [ ]:
from furax.obs.pointing import PointingOperator

# frame='boresight' → point into the boresight (telescope) frame
# frame='detector'  → already in detector frame (no HWP)
pointing = PointingOperator.create(
    landscape,
    qbore,
    qdet,
    chunk_size=16,
    frame='boresight',   # for HWP instruments
    interpolate=False,   # True → bilinear interpolation
)

# Forward: sky map → TOD (scatter)
tod_from_pointing = pointing(sky_true)       # StokesIQU → (NDET, NSAMP) IQU stokes pytree
# Adjoint: TOD → sky map (accumulate)
sky_from_pointing = pointing.T(jnp.ones((NDET, NSAMP)))

print('pointing.in_structure :', pointing.in_structure)    # StokesIQU structure
print('pointing.out_structure:', pointing.out_structure)

### HWP operator

`HWPOperator` models an ideal half-wave plate. Its Mueller matrix is $\mathrm{diag}(1, 1, -1, -1)$
(flips the sign of U and V). For a rotating HWP at angle $\chi$, it applies
$R(-2\chi) \circ \mathrm{HWP} \circ R(2\chi)$ — a rotation by $4\chi$ in QU space.

In [ ]:
from furax.obs.operators import HWPOperator

# HWP with angles — wraps the bare HWP inside R(-theta)@HWP@R(theta)
hwp = HWPOperator.create(
    shape=(NDET, NSAMP),
    dtype=jnp.float64,
    stokes='IQU',
    angles=hwp_angles,  # shape (NSAMP,), broadcast over detectors
)

# Apply to a sky-shaped pytree (after pointing has projected it to TOD shape)
from furax.obs.stokes import StokesIQU
tod_stokes = StokesIQU(
    i=jnp.ones((NDET, NSAMP)),
    q=jnp.zeros((NDET, NSAMP)),
    u=jnp.zeros((NDET, NSAMP)),
)
modulated = hwp(tod_stokes)   # I stays, Q/U are rotated by 4*chi
print('HWP(I=1, Q=0, U=0): I=', float(modulated.i[0, 0]),
      'Q=', float(modulated.q[0, 0]),
      'U=', float(modulated.u[0, 0]))

### Linear polarizer operator

`LinearPolarizerOperator` extracts the scalar signal seen by a polarization-sensitive
detector aligned with the x-axis: $d = \tfrac{1}{2}(I + Q)$.
For a detector at angle $\psi$, it first applies `QURotation(psi)` to rotate
the Q/U frame, then projects.

In [ ]:
from furax.obs.operators import LinearPolarizerOperator

# Polarizer with no angle offset (x-aligned)
polarizer = LinearPolarizerOperator.create(
    shape=(NDET, NSAMP),
    dtype=jnp.float64,
    stokes='IQU',
)

# StokesIQU → Array(NDET, NSAMP)
d = polarizer(tod_stokes)   # d = 0.5*(I + Q) = 0.5
print('polarizer(I=1, Q=0, U=0):', float(d[0, 0]))  # 0.5

### Full acquisition chain

The `build_acquisition_operator` convenience function assembles the chain
and calls `.reduce()` to simplify via algebraic rewrite rules
(e.g. `LinearPolarizer @ HWP → LinearPolarizer`).

In [ ]:
from furax.mapmaking.acquisition import build_acquisition_operator

# With HWP: chain = Polarizer @ QURotation(gamma) @ HWP(chi) @ Pointing
acq_hwp = build_acquisition_operator(
    landscape,
    boresight_quaternions=qbore,
    detector_quaternions=qdet,
    hwp_angles=hwp_angles,
    pointing_on_the_fly=True,
    pointing_chunk_size=16,
    dtype=jnp.float64,
)

# Without HWP: chain = Polarizer @ Pointing
acq_no_hwp = build_acquisition_operator(
    landscape,
    boresight_quaternions=qbore,
    detector_quaternions=qdet,
    # hwp_angles omitted
    dtype=jnp.float64,
)

# Forward: sky (StokesIQU) → TOD (ndets, nsamps)
sim_tod = acq_hwp(sky_true)
print('Simulated TOD shape:', sim_tod.shape)  # (NDET, NSAMP)

# Adjoint: TOD → sky (dirty map)
dirty_map = acq_hwp.T(sim_tod)
print('Dirty map type:', type(dirty_map).__name__)  # StokesIQU

In [ ]:
# Demodulated acquisition (post-HWP processing)
# The HWP modulation is already removed from the TOD.
# The acquisition operator becomes: 0.5 * QURotation(-gamma) @ Pointing

acq_demod = build_acquisition_operator(
    landscape,
    boresight_quaternions=qbore,
    detector_quaternions=qdet,
    hwp_angles=None,
    demodulated=True,
    dtype=jnp.float64,
)

# Demodulated TOD has Stokes structure
from furax.obs.stokes import StokesIQU
demod_tod = StokesIQU(
    i=jnp.ones((NDET, NSAMP)),
    q=jnp.zeros((NDET, NSAMP)),
    u=jnp.zeros((NDET, NSAMP)),
)
demod_sky = acq_demod.T(demod_tod)  # dirty map
print('Demod dirty map:', type(demod_sky).__name__)

## 2.2 Noise Models

FURAX provides two noise models.

### White noise

$$N = \mathrm{diag}(\sigma_1^2, \ldots, \sigma_d^2) \otimes \mathbf{1}_{n_s}$$

Each detector has a constant variance $\sigma_d^2$ across all samples.
The inverse is also diagonal.

### Atmospheric (1/f) noise

$$S(f) = \sigma^2 \left[1 + \left(\frac{f + f_0}{f_k}\right)^\alpha\right]$$

Four parameters per detector: $\sigma$ (white level), $\alpha$ (spectral index,
typically $\alpha < 0$), $f_k$ (knee frequency), $f_0$ (minimum frequency offset).
The noise covariance is a circulant matrix → represented as a
`SymmetricBandToeplitzOperator` truncated at a user-specified correlation length.

In [ ]:
from furax.mapmaking.noise import WhiteNoiseModel, AtmosphericNoiseModel

SAMPLE_RATE = 200.0   # Hz

# White noise model
sigma = jnp.array([0.01, 0.012, 0.011])  # per-detector RMS (NDET,)
white = WhiteNoiseModel(sigma=sigma)
print('WhiteNoiseModel sigma:', white.sigma)

# Atmospheric 1/f noise model
atm = AtmosphericNoiseModel(
    sigma=sigma,
    alpha=jnp.array([-3.5, -3.5, -3.5]),  # spectral index
    fk=jnp.array([0.1, 0.1, 0.1]),        # knee frequency (Hz)
    f0=jnp.array([1e-5, 1e-5, 1e-5]),     # minimum frequency offset (Hz)
)

# Evaluate PSD at given frequencies
freqs = jnp.logspace(-2, jnp.log10(SAMPLE_RATE / 2), 500)
psd_white = white.psd(freqs)   # (NDET, n_freqs)
psd_atm   = atm.psd(freqs)

fig, ax = plt.subplots(figsize=(7, 3))
for i in range(NDET):
    ax.loglog(freqs, psd_atm[i], label=f'det {i} (1/f)')
ax.loglog(freqs, psd_white[0], 'k--', label='white')
ax.axvline(float(atm.fk[0]), color='gray', ls=':', label='$f_k$')
ax.set_xlabel('Frequency [Hz]'); ax.set_ylabel('PSD')
ax.legend(); ax.set_title('Noise PSD')
plt.tight_layout(); plt.show()

### Fitting noise models from data

Both models expose a `.fit_psd_model()` classmethod that fits to Welch-estimated PSDs.
The white noise level is estimated from the high-frequency tail. The 1/f parameters
are fitted by minimising the Whittle log-likelihood via L-BFGS-B.

In [ ]:
from furax.mapmaking.config import NoiseFitConfig

# Simulate 1/f TOD for demonstration
key = jax.random.PRNGKey(42)
from furax.mapmaking.gap_filling import generate_noise_realization
from furax import SymmetricBandToeplitzOperator

# Estimate PSD via Welch's method
fake_tod = jax.random.normal(key, (NDET, 4096))
f, Pxx = jax.scipy.signal.welch(fake_tod, fs=SAMPLE_RATE, nperseg=512)
print('Welch frequencies:', f.shape, '  PSD:', Pxx.shape)  # (NDET, n_freqs)

fit_cfg = NoiseFitConfig(
    nperseg=512,
    mask_hwp_harmonics=True,
    high_freq_nyquist=0.02,
)
hwp_freq = jnp.array(2.0)  # HWP rotation rate in Hz

# Fit white noise model
fitted_white = WhiteNoiseModel.fit_psd_model(
    f, Pxx, sample_rate=jnp.array(SAMPLE_RATE), hwp_frequency=hwp_freq, config=fit_cfg
)
print('Fitted white sigma:', fitted_white.sigma)

# Fit atmospheric (1/f) model
fitted_atm = AtmosphericNoiseModel.fit_psd_model(
    f, Pxx, sample_rate=jnp.array(SAMPLE_RATE), hwp_frequency=hwp_freq, config=fit_cfg
)
print('Fitted atm sigma :', fitted_atm.sigma)
print('Fitted fk        :', fitted_atm.fk)

### Noise operators: diagonal vs Toeplitz

The `.operator()` and `.inverse_operator()` methods return the appropriate linear
operator for applying $N$ or $N^{-1}$ to a TOD vector.

- `WhiteNoiseModel` → `DiagonalOperator` (variance $\sigma_d^2$ per sample)
- `AtmosphericNoiseModel` → `SymmetricBandToeplitzOperator` truncated at `correlation_length`

The Toeplitz operator applies the filter in $O(n_s \log n_s)$ via FFT convolution.

In [ ]:
from furax import DiagonalOperator, SymmetricBandToeplitzOperator

data_struct = jax.ShapeDtypeStruct((NDET, 4096), jnp.float64)
CORR_LEN = 1000  # Toeplitz bandwidth in samples

# White noise inverse
N_inv_white = white.inverse_operator(data_struct)
print(type(N_inv_white).__name__)   # DiagonalOperator

# Atmospheric noise covariance and inverse
N_atm = atm.operator(
    data_struct,
    sample_rate=SAMPLE_RATE,
    correlation_length=CORR_LEN,
)
N_inv_atm = atm.inverse_operator(
    data_struct,
    sample_rate=SAMPLE_RATE,
    correlation_length=CORR_LEN,
)
print(type(N_atm).__name__)      # SymmetricBandToeplitzOperator
print('Band shape:', N_inv_atm.band_values.shape)  # (NDET, CORR_LEN)

# Applying N^{-1} to a TOD
test_tod = jax.random.normal(key, (NDET, 4096))
filtered = N_inv_atm(test_tod)   # (NDET, 4096)
print('Filtered TOD shape:', filtered.shape)

## 2.3 The Mapmaking Equation

The optimal (maximum likelihood) sky map estimate satisfies:

$$\underbrace{\left(\mathbf{H}^\top \mathbf{N}^{-1} \mathbf{H}\right)}_{\mathbf{A}} \, \hat{s}
= \underbrace{\mathbf{H}^\top \mathbf{N}^{-1} d}_{b}$$

For multiple observations:

$$\mathbf{A} = \sum_i \mathbf{H}_i^\top \mathbf{N}_i^{-1} \mathbf{H}_i, \qquad
b = \sum_i \mathbf{H}_i^\top \mathbf{N}_i^{-1} d_i$$

For white noise, $\mathbf{A}$ is block-diagonal in pixel space (the binned case).
For correlated noise, it mixes pixels and must be inverted iteratively with PCG.

### System matrix

In [ ]:
# Toy demonstration: build A = H^T N^{-1} H and compute the dirty map b = H^T N^{-1} d

# Simulate a noisy observation
key = jax.random.PRNGKey(7)
k1, k2 = jax.random.split(key)
noise_tod = float(sigma[0]) * jax.random.normal(k1, (NDET, NSAMP))
obs_tod = sim_tod + noise_tod

# Build system matrix using white noise (binned case)
data_struct_small = jax.ShapeDtypeStruct((NDET, NSAMP), jnp.float64)
N_inv_w = white.inverse_operator(data_struct_small)

A = (acq_hwp.T @ N_inv_w @ acq_hwp).reduce()
b = (acq_hwp.T @ N_inv_w)(obs_tod)   # dirty map = H^T N^{-1} d

print('b.i shape:', b.i.shape)  # (n_pixels,)

### Block-Jacobi Preconditioner

`BJPreconditioner` computes the dense per-pixel Stokes covariance matrix by probing
$\mathbf{A}$ with the Stokes basis vectors. For the binned case with white noise,
this IS the exact inverse — so the PCG converges in one step.
For the ML case, it is an approximation based on the diagonal noise variance,
used only as a preconditioner to accelerate convergence.

In [ ]:
from furax.mapmaking.preconditioner import BJPreconditioner

# Create BJ preconditioner from the system matrix
BJ = BJPreconditioner.create(A)

# Get the dense per-pixel blocks as array
blocks = BJ.get_blocks()   # (n_pixels, n_stokes, n_stokes)
print('BJ blocks shape:', blocks.shape)

# Inverse preconditioner (M^{-1} applied pixel-by-pixel)
M_inv = BJ.inverse()

# For the binned case with white noise, M_inv(b) IS the solution
sol_binned = M_inv(b)
print('Binned solution .i shape:', sol_binned.i.shape)

### PCG Solver

FURAX delegates the iterative solve to `lineax.linear_solve`. The solver, tolerance,
and preconditioner are configured via `SolverConfig` in the mapmaking config.

For the ML case (correlated noise), the PCG loop is:
```
r ← b - A @ x
z ← M^{-1} @ r          ← preconditioned residual
x ← x + alpha * p
... (standard CG update)
```

Pixels that are masked (low hits or poor conditioning) are excluded before solving
to keep the system well-conditioned.

In [ ]:
import lineax
from furax import OperatorTag
from furax.interfaces.lineax import as_lineax_operator

# Wrap for lineax (tags SPD so CG is used)
spd = OperatorTag.POSITIVE_SEMIDEFINITE
lx_system = as_lineax_operator(A, spd)
lx_precond = as_lineax_operator(M_inv, spd)

# Configure solver via SolverConfig
# config.solver = SolverConfig(rtol=1e-6, atol=0, max_steps=1000)
solver = lineax.CG(rtol=1e-6, atol=0, max_steps=200)

# Initial guess: apply preconditioner to the RHS
x0 = M_inv(b)

solution = lineax.linear_solve(
    lx_system,
    b,
    solver=solver,
    options={'preconditioner': lx_precond, 'y0': x0},
    throw=False,  # don't raise on non-convergence
)

recovered = solution.value
print('Steps to converge:', solution.stats['num_steps'])
print('I-map residual RMS:', float(jnp.std(recovered.i - sky_true.i)))

In [ ]:
# SolverConfig in MapMakingConfig
from furax.mapmaking.config import SolverConfig

# Adjust convergence settings
ml_config_tight = MapMakingConfig(
    method=Methods.MAXL,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=64)),
    noise=NoiseConfig(
        white=False,                  # ML requires correlated noise model
        fit_from_data=True,
        correlation_length=1000,      # Toeplitz bandwidth in samples
        fitting=NoiseFitConfig(nperseg=512),
    ),
    solver=SolverConfig(
        rtol=1e-8,      # tighter convergence
        atol=0,
        max_steps=500,
    ),
    pointing=PointingConfig(on_the_fly=True),
)
print('ML config: correlation_length =', ml_config_tight.noise.correlation_length)

## 2.4 Gap Filling and Nested PCG

### Gap filling

Flagged TOD samples (glitches, turnarounds) create gaps in the data stream.
Naively zeroing these gaps introduces a spectral bias: the Toeplitz noise model
assumes stationary data, and discontinuities at gap edges corrupt the filter.

FURAX fills gaps with a **constrained noise realisation** that is statistically
consistent with the surrounding data (Stompor et al. 2002, §II.C):

$$y = \xi + \mathbf{N} \mathbf{P}^\top (\mathbf{P N P}^\top)^{-1} \mathbf{P}(d - \xi)$$

where $\mathbf{P}$ selects valid samples, and $\xi$ is a noise realization drawn from $\mathbf{N}$.
The valid samples are preserved exactly: $y_{\rm valid} = d_{\rm valid}$.

Gap filling is enabled by default (`gaps.fill=True`) for the ML mapmaker.

In [ ]:
from furax.mapmaking.gap_filling import GapFillingOperator
from furax import IndexOperator

# Toy example: 1-detector TOD with 20% gaps
n_samp_gf = 1024
key = jax.random.PRNGKey(0)
k1, k2 = jax.random.split(key)
gf_tod = jax.random.normal(k1, (1, n_samp_gf), dtype=jnp.float64)

# Create a gap mask (True = valid)
gap_mask = jax.random.uniform(k2, (n_samp_gf,)) > 0.2   # 80% valid
print(f'Valid samples: {int(jnp.sum(gap_mask))} / {n_samp_gf}')

# Noise covariance (white for simplicity; use SymmetricBandToeplitz for 1/f)
gf_struct = jax.ShapeDtypeStruct((1, n_samp_gf), jnp.float64)
sigma_gf = jnp.array([0.01])
N_gf = WhiteNoiseModel(sigma_gf).operator(gf_struct)
N_inv_gf = WhiteNoiseModel(sigma_gf).inverse_operator(gf_struct)

# Build the gap selector
pack = IndexOperator(jnp.where(gap_mask), in_structure=gf_struct)

gf = GapFillingOperator(
    cov=N_gf,
    mask_op=pack,
    icov=N_inv_gf,        # used as preconditioner for the inner CG solve
    rate=SAMPLE_RATE,
    max_cg_steps=50,
    rtol=1e-4,
)

gf_key = jax.random.key(42)
filled_tod = gf(gf_key, gf_tod)    # gaps replaced, valid samples unchanged

# Verify valid samples are untouched
assert jnp.allclose(filled_tod[0, gap_mask], gf_tod[0, gap_mask])
print('Valid samples unchanged:', jnp.allclose(filled_tod[0, gap_mask], gf_tod[0, gap_mask]))

In [ ]:
from furax.mapmaking.config import GapFillingConfig, GapsConfig

# Gap-filling options in MapMakingConfig
ml_with_gapfill = MapMakingConfig(
    method=Methods.MAXL,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=64)),
    noise=NoiseConfig(white=False, fit_from_data=True, correlation_length=1000,
                      fitting=NoiseFitConfig(nperseg=512)),
    gaps=GapsConfig(
        fill=True,                    # enable gap filling (default True for ML)
        fill_options=GapFillingConfig(
            seed=286502183,           # PRNG seed for reproducibility
            max_steps=50,             # inner CG iteration budget
            rtol=1e-4,                # inner CG convergence
        ),
        nested_pcg=False,             # see below
    ),
    pointing=PointingConfig(on_the_fly=True),
)
print('Gap filling enabled:', ml_with_gapfill.gaps.fill)

### Nested PCG

An alternative to gap filling. Instead of using the truncated Toeplitz approximation
$N^{-1}$ directly, the outer PCG loop solves:

$$\mathbf{H}^\top \underbrace{\mathbf{M}(\mathbf{N M})^{-1} \mathbf{M}}_{\approx N^{-1}} \mathbf{H} \hat{s} = \ldots$$

where the inner $(\mathbf{N M})^{-1}$ is solved by a **nested** CG call at each outer iteration.
This is more accurate but roughly doubles the wall time.

Enable with `gaps.nested_pcg=True`.

In [ ]:
ml_nested = MapMakingConfig(
    method=Methods.MAXL,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=64)),
    noise=NoiseConfig(white=False, fit_from_data=True, correlation_length=1000,
                      fitting=NoiseFitConfig(nperseg=512)),
    gaps=GapsConfig(
        fill=False,       # gap filling is skipped when nested_pcg=True
        nested_pcg=True,  # inner CG to invert the noise
    ),
    solver=SolverConfig(rtol=1e-6, atol=0, max_steps=200),
    pointing=PointingConfig(on_the_fly=True),
)
print('Nested PCG:', ml_nested.gaps.nested_pcg)

---
# Part 3: Mapmaking Algorithms

FURAX implements four mapmaking algorithms through the `MapMaker` class hierarchy,
each with different assumptions about the noise and systematics.

## 3.1 BinnedMapMaker

**When to use:** White (uncorrelated) noise; fast, one-shot solution.

**Algorithm:**

Since $N = \mathrm{diag}(\sigma_d^2)$, the system matrix $A = H^\top N^{-1} H$ is
block-diagonal in pixel space (one $n_s \times n_s$ block per pixel, where $n_s$ is
the number of Stokes components). The solution is:

$$\hat{s} = \left( H^\top N^{-1} H \right)^{-1} H^\top N^{-1} d$$

No iterations are needed: the per-pixel blocks are inverted analytically.
The PCG formally converges in one step.

**Key configuration:** `noise.white=True` (required).

In [ ]:
from furax.mapmaking.mapmaker import BinnedMapMaker

config_binned = MapMakingConfig(
    method=Methods.BINNED,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    noise=NoiseConfig(
        white=True,          # REQUIRED for BinnedMapMaker
        fit_from_data=True,
        fitting=NoiseFitConfig(nperseg=512),
    ),
    pointing=PointingConfig(on_the_fly=True),
    scanning_mask=False,
    hits_cut=1e-2,
    cond_cut=1e-2,
)

# Use MapMaker.from_config for dispatch, or instantiate directly
binned_maker = MapMaker.from_config(config_binned)
assert isinstance(binned_maker, BinnedMapMaker)

binned_result = binned_maker.make_map(toast_obs)
print('Binned map shape:', binned_result['map'].shape)     # (3, n_pixels)
print('Weights shape   :', binned_result['weights'].shape) # (n_pixels, 3, 3)

In [ ]:
# Walk through what BinnedMapMaker.make_map() does internally

# 1. Get landscape (from config or by scanning observation)
landscape_b = binned_maker.get_landscape(toast_obs)
print('Landscape:', type(landscape_b).__name__, 'nside=', landscape_b.nside)

# 2. Acquisition operator H
H = binned_maker.get_acquisition(toast_obs, landscape_b)
print('H type:', type(H).__name__)

# 3. Noise model
noise_m = binned_maker.get_or_fit_noise_model(toast_obs)
print('Noise:', type(noise_m).__name__, 'sigma_mean=', float(jnp.mean(noise_m.sigma)))

# 4. System matrix (BJ is exactly the solution for white noise)
data_b = toast_obs.get_tods().astype(jnp.float64)
N_inv_b = noise_m.inverse_operator(jax.ShapeDtypeStruct(data_b.shape, data_b.dtype))
A_b = (H.T @ N_inv_b @ H).reduce()
BJ_b = BJPreconditioner.create(A_b)

# 5. Map = (H^T N^{-1} H)^{-1} H^T N^{-1} d
b_b = (H.T @ N_inv_b)(data_b)
sol_b = BJ_b.inverse()(b_b)
print('I-map RMS (direct):', float(jnp.std(sol_b.i)))

## 3.2 MLMapmaker (Maximum Likelihood)

**When to use:** Correlated (1/f atmospheric) noise; requires iterative solver.

**Algorithm:**

With the full Toeplitz $N$, the system $A \hat{s} = b$ is solved by PCG.
The preconditioner uses the diagonal (white noise) approximation:

$$M = \left( H^\top N_{\rm diag}^{-1} H \right)^{-1}$$

A pixel selection step first removes poorly-observed pixels (few hits or poor
Stokes conditioning) to keep the system well-posed.

**Key configuration:** `noise.white=False` (required), `noise.correlation_length`.

In [ ]:
from furax.mapmaking.mapmaker import MLMapmaker

config_ml = MapMakingConfig(
    method=Methods.MAXL,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    noise=NoiseConfig(
        white=False,                  # REQUIRED for MLMapmaker
        fit_from_data=True,
        correlation_length=200,       # Toeplitz bandwidth (smaller for speed)
        fitting=NoiseFitConfig(nperseg=512),
    ),
    solver=SolverConfig(rtol=1e-4, atol=0, max_steps=100),
    pointing=PointingConfig(on_the_fly=True),
    scanning_mask=False,
    hits_cut=1e-2,
    cond_cut=1e-2,
    gaps=GapsConfig(fill=True, fill_options=GapFillingConfig(max_steps=20, rtol=1e-3)),
)

ml_maker = MapMaker.from_config(config_ml)
assert isinstance(ml_maker, MLMapmaker)

ml_result = ml_maker.make_map(toast_obs)
print('ML map shape :', ml_result['map'].shape)
print('noise_fit    :', ml_result.get('noise_fit', 'not stored'))

In [ ]:
# MLMapmaker supports template subtraction for ground pickup / HWP systematics
from furax.mapmaking.config import TemplatesConfig, _PolyTemplateConfig, _HWPSynchronousTemplateConfig

config_ml_tmpl = MapMakingConfig(
    method=Methods.MAXL,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    noise=NoiseConfig(white=False, fit_from_data=True, correlation_length=200,
                      fitting=NoiseFitConfig(nperseg=512)),
    solver=SolverConfig(rtol=1e-4, max_steps=100),
    pointing=PointingConfig(on_the_fly=True),
    templates=TemplatesConfig(
        polynomial=_PolyTemplateConfig(max_poly_order=3),        # per-scan polynomials
        hwp_synchronous=_HWPSynchronousTemplateConfig(n_harmonics=3), # HWP harmonics
        # scan_synchronous=_ScanSynchronousTemplateConfig(),     # az-synchronous templates
        # azhwp_synchronous=_AzimuthHWPSynchronousTemplateConfig(),
        regularization=0.0,  # optional L2 regularization on template amplitudes
    ),
    gaps=GapsConfig(fill=True),
)
print('Templates enabled:', config_ml_tmpl.use_templates)
print('Template types:', [
    k for k, v in [
        ('polynomial', config_ml_tmpl.templates.polynomial),
        ('hwp_synchronous', config_ml_tmpl.templates.hwp_synchronous),
    ] if v is not None
])

## 3.3 ATOPMapMaker

**When to use:** Alternative 1/f noise removal; QU-only output; faster than full ML.

**Algorithm (ATOP = "Alternating Temporal Operations and Projections"):**

Instead of applying the full $N^{-1}$ (Toeplitz), ATOP applies a binning
projection $F^\top F$ where $F$ averages consecutive $\tau$-sample blocks.
This suppresses 1/f noise without the cost of computing a Toeplitz inverse:

$$A = H^\top M F^\top (\mathrm{diag}^{-1}) F M H$$

**Constraints:**
- Requires `noise.white=True` (binned approximation for preconditioner)
- Only supports Stokes `'QU'` (I is not reconstructed)
- Requires `atop_tau >= 2` (block size)
- `stokes='IQU'` is automatically downgraded to `'QU'`

In [ ]:
from furax.mapmaking.mapmaker import ATOPMapMaker

config_atop = MapMakingConfig(
    method=Methods.ATOP,
    atop_tau=10,                          # block size tau (must be >= 2)
    landscape=LandscapeConfig(
        stokes='QU',                      # ATOP only supports QU
        healpix=HealpixConfig(nside=16),
    ),
    noise=NoiseConfig(
        white=True,                       # REQUIRED (diagonal preconditioner)
        fit_from_data=True,
        fitting=NoiseFitConfig(nperseg=512),
    ),
    solver=SolverConfig(rtol=1e-4, max_steps=100),
    pointing=PointingConfig(on_the_fly=True),
    hits_cut=1e-2,
    cond_cut=1e-2,
)

atop_maker = MapMaker.from_config(config_atop)
assert isinstance(atop_maker, ATOPMapMaker)

atop_result = atop_maker.make_map(toast_obs)
print('ATOP map shape:', atop_result['map'].shape)  # (2, n_pixels) — Q and U only

In [ ]:
# ATOP is also supported in MultiObservationMapMaker
multi_atop_config = MapMakingConfig(
    method=Methods.ATOP,
    atop_tau=10,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    # stokes='IQU' is automatically downgraded to 'QU' with a log warning
    noise=NoiseConfig(white=True, fit_from_data=True, fitting=NoiseFitConfig(nperseg=512)),
    pointing=PointingConfig(on_the_fly=True),
)

maker_atop = MultiObservationMapMaker(toast_lazy_obs, config=multi_atop_config)
print('Downgraded stokes:', maker_atop.config.landscape.stokes)  # 'QU'

results_atop = maker_atop.run()
print('icov shape:', results_atop.icov.shape)  # (2, 2, n_pixels)

## 3.4 TwoStepMapmaker

**When to use:** Systematic templates + white noise; faster than ML with templates.

**Algorithm:**

Jointly estimates template amplitudes $a$ and the sky map $s$:

$$d = H s + F a + n, \qquad N = \mathrm{diag}(\sigma_d^2)$$

Using the fact that $A = H^\top N^{-1} H$ is easy to invert (white noise), the
estimator decouples into two steps:

1. **Estimate templates:** $\hat{a} = (F^\top P_\perp F)^{-1} F^\top P_\perp d$
   where $P_\perp = N^{-1}(1 - H A^{-1} H^\top N^{-1})$ is the sky-projection operator.

2. **Estimate map:** $\hat{s} = A^{-1} H^\top N^{-1}(d - F\hat{a})$

The template amplitudes are estimated via PCG on the (typically small) template-template
system, not the full pixel-space system.

**Key configuration:** `noise.white=True` (required), templates must be non-empty.

In [ ]:
from furax.mapmaking.mapmaker import TwoStepMapmaker
from furax.mapmaking.config import _ScanSynchronousTemplateConfig

config_twostep = MapMakingConfig(
    method=Methods.TWOSTEP,
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    noise=NoiseConfig(
        white=True,       # REQUIRED for TwoStep
        fit_from_data=True,
        fitting=NoiseFitConfig(nperseg=512),
    ),
    solver=SolverConfig(rtol=1e-4, max_steps=50),
    pointing=PointingConfig(on_the_fly=True),
    templates=TemplatesConfig(
        polynomial=_PolyTemplateConfig(max_poly_order=3),
        # TwoStep supports all the same templates as MLMapmaker
    ),
)

twostep_maker = MapMaker.from_config(config_twostep)
assert isinstance(twostep_maker, TwoStepMapmaker)

twostep_result = twostep_maker.make_map(toast_obs)
print('TwoStep map shape        :', twostep_result['map'].shape)
print('Template amplitude keys  :', [k for k in twostep_result if k.startswith('template')])

## 3.5 Creating Your Own Mapmaker

All four algorithms share the same base class `MapMaker` (a frozen dataclass holding
a `MapMakingConfig`). To implement a custom algorithm:

1. Subclass `MapMaker`
2. Implement `make_map(self, observation) -> dict`
3. Use the helper methods on `MapMaker` to reuse standard components:
   - `get_landscape(obs)` — determine output pixelization
   - `get_acquisition(obs, landscape)` — build H
   - `get_or_fit_noise_model(obs)` — white or 1/f noise
   - `get_pixel_selector(blocks, landscape)` — mask low-hit pixels
   - `get_mask_projector(obs)` — combined scanning + sample mask
   - `get_template_operator(obs)` — build template block-row operator

In [ ]:
from dataclasses import dataclass
from typing import Any

import jax
import numpy as np

from furax.mapmaking._observation import AbstractGroundObservation
from furax.mapmaking.mapmaker import MapMaker
from furax.mapmaking.preconditioner import BJPreconditioner


@dataclass
class HalfIterMLMapMaker(MapMaker):
    """One-step ML approximation: apply M^{-1} b instead of solving the full system.

    Equivalent to a single PCG iteration starting from zero.
    Useful as a quick diagnostic or warm-start estimate.
    """

    def make_map(self, observation: AbstractGroundObservation[Any]) -> dict[str, Any]:
        config = self.config

        # --- Standard setup (reusing MapMaker helpers) ---
        data = observation.get_tods().astype(config.dtype)
        landscape = self.get_landscape(observation)
        H = self.get_acquisition(observation, landscape)
        masker = self.get_mask_projector(observation)
        noise_model = self.get_or_fit_noise_model(observation)
        data_struct = jax.ShapeDtypeStruct(data.shape, data.dtype)
        N_inv = noise_model.inverse_operator(
            data_struct,
            sample_rate=observation.sample_rate,
            correlation_length=config.noise.correlation_length,
        )

        # --- Build diagonal preconditioner using white noise approximation ---
        sigma = noise_model.to_white_noise_model().sigma if hasattr(noise_model, 'to_white_noise_model') else noise_model.sigma
        from furax.mapmaking.noise import WhiteNoiseModel
        diag_N_inv = WhiteNoiseModel(sigma).inverse_operator(data_struct)
        diag_A = BJPreconditioner.create((H.T @ diag_N_inv @ masker @ H).reduce())

        # --- Pixel selection ---
        selector = self.get_pixel_selector(diag_A.get_blocks(), landscape)

        # --- One-step approximation: M^{-1} b ---
        b = (H.T @ masker @ N_inv)(data)    # RHS: H^T N^{-1} d
        approx_map = (selector @ diag_A.inverse() @ selector.T)(b)

        return {
            'map': np.array(jax.tree.leaves(approx_map)),
            'weights': np.array(diag_A.get_blocks()),
        }


# Use just like any other mapmaker
# The method field is not checked for custom subclasses
from furax.mapmaking.config import Methods

custom_config = MapMakingConfig(
    method=Methods.MAXL,    # unused by our class but required by MapMakingConfig
    landscape=LandscapeConfig(stokes='IQU', healpix=HealpixConfig(nside=16)),
    noise=NoiseConfig(white=False, fit_from_data=True, correlation_length=200,
                      fitting=NoiseFitConfig(nperseg=512)),
    pointing=PointingConfig(on_the_fly=True),
)

custom_maker = HalfIterMLMapMaker(config=custom_config)
custom_result = custom_maker.make_map(toast_obs)
print('Custom map shape:', custom_result['map'].shape)

### Extending MultiObservationMapMaker

For multi-observation pipelines, subclass `MultiObservationMapMaker` and override
`make_maps()`. The main building blocks are already available as methods:

```python
class MyMultiObsMaker(MultiObservationMapMaker):
    def make_maps(self) -> MapMakingResults:
        model = self.build_model()          # stacked ObservationModel
        hits  = self.accumulate_hits(model) # per-pixel hit count
        rhs   = self.accumulate_rhs(model)  # H^T N^{-1} d summed over observations
        # ... custom solve ...
        return MapMakingResults(map=estimate, ...)
```

`ObservationModel` holds the operators for each observation:
- `.H` — acquisition operator
- `.W` — inverse noise covariance operator
- `.masker` — sample mask projector
- `.apply_system(x)` — evaluates $H^\top W H x$ for one observation
- `.rhs(data, config)` — evaluates $H^\top W d$ (with optional gap filling)

---
## Summary

### Choosing an algorithm

| Situation | Recommended algorithm |
|-----------|----------------------|
| Fast, noise characterisation, white noise assumption | `Methods.BINNED` |
| Maximum fidelity, 1/f noise, many PCG iterations affordable | `Methods.MAXL` |
| 1/f noise removal without full Toeplitz inversion | `Methods.ATOP` |
| Ground pickup / HWP harmonics + white noise | `Methods.TWOSTEP` |
| 1/f noise + systematics | `Methods.MAXL` with templates |

### Key API recap

```python
# Single observation
maker = MapMaker.from_config(config)         # or MapMaker.from_yaml(path)
results_dict = maker.run(obs, out_dir=None)  # dict with 'map', 'weights', ...

# Multiple observations
maker = MultiObservationMapMaker(lazy_obs_list, config=config)
results = maker.run(out_dir=None)            # MapMakingResults dataclass
results.map.i   # I sky map array
results.icov    # (n_stokes, n_stokes, *pixels) inverse covariance
results.save('/output/dir/')
```